# H7 · Qwen3-4B (4-bit) in place of Qwen3-8B: claims, explanations, LLM judge (Himasri)

Runs the team's own scripts unchanged, with only the model swapped (`src/eval/swap_model.py`), and compares with the Qwen3-8B results already in the repo:
1. **Claim extraction** (Youssef's `src.claims.evaluate`, 200 held-out messages) vs 8B F1 0.984
2. **Cited explanations + faithfulness** (Rathi's `src.explain.generate` + `faithfulness`, same 250 claims, seed 42) vs 8B faithfulness 0.32
3. **LLM-judge baseline** (Feba's `src.eval.baselines.llm_judge`, MultiFC test + twins, AVeriTeC) vs 8B macro-F1 0.233

All 4B outputs go to `results/H7/qwen4b/`; nothing from 8B is overwritten.

**Before you start:** Runtime → **T4 GPU** (L4 faster); Colab secret `GH_TOKEN`. Time on a T4: ~1–1.5 h (the LLM judge on 3,490 records is the longest step).

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/himaallu/Group-Project.git /content/twincheck
%cd /content/twincheck
!git checkout -q himasri/H7-qwen4b-swap
!pip install -q peft bitsandbytes accelerate
from google.colab import userdata
import os
os.makedirs('results/H7/qwen4b', exist_ok=True)
SWAP = 'python -u -m src.eval.swap_model --model Qwen/Qwen3-4B --module'

## 2. 10-row checks (outputs in /content/test10)

In [ ]:
!{SWAP} src.eval.baselines.llm_judge -- --predict data/processed/en/multifc_test.jsonl --limit 10 --out /content/test10/judge.jsonl --no-wandb 2>&1 | tail -2
!{SWAP} src.explain.generate -- --n-en 3 --n-ar 2 --out /content/test10/notes.jsonl 2>&1 | tail -2
!head -c 600 /content/test10/notes.jsonl

## 3. Claim extraction (200 held-out messages)
Youssef's evaluator writes to `results/Y3/`; the 4B files are moved to `results/H7/qwen4b/` and his 8B files restored.

In [ ]:
!{SWAP} src.claims.evaluate -- --backend qwen --no_wandb 2>&1 | tail -4
!mv results/Y3/f1_qwen.json results/H7/qwen4b/claims_f1_qwen4b.json
!mv results/Y3/preds_qwen.jsonl results/H7/qwen4b/claims_preds_qwen4b.jsonl
!git checkout -- results/Y3

## 4. Cited explanations (same 250 claims as the 8B run) + faithfulness

In [ ]:
!{SWAP} src.explain.generate -- --out results/H7/qwen4b/notes.jsonl 2>&1 | tail -3
!python -m src.explain.faithfulness --notes results/H7/qwen4b/notes.jsonl --out-dir results/H7/qwen4b/explain 2>&1 | tail -3

## 5. LLM judge (MultiFC test + twins, then AVeriTeC; resumes after a disconnect)

In [ ]:
!{SWAP} src.eval.baselines.llm_judge -- --predict data/processed/en/multifc_test.jsonl data/copies/en/test.jsonl --out results/H7/qwen4b/preds_llm_judge4b.jsonl --no-wandb 2>&1 | tail -2
!{SWAP} src.eval.baselines.llm_judge -- --predict data/processed/en/averitec_test.jsonl --out results/H7/qwen4b/preds_avt_llm_judge4b.jsonl --no-wandb 2>&1 | tail -2

## 6. 4B vs 8B summary (paste this into the chat with Claude)

In [ ]:
import json
from src.eval.loaders import load_jsonl, load_preds, join_gold
from src.eval.metrics import compute_all
c8, c4 = json.load(open('results/Y3/f1_qwen.json')), json.load(open('results/H7/qwen4b/claims_f1_qwen4b.json'))
print('Claim extraction F1   8B / 4B:', {k: (round(c8[k]['f1'], 3), round(c4[k]['f1'], 3)) for k in c8})
e8, e4 = json.load(open('results/R5/explain_eval.json')), json.load(open('results/H7/qwen4b/explain/explain_eval.json'))
for k in ('faithfulness', 'citation_accuracy', 'share_notes_fully_faithful'):
    print(f'Explanations {k:26} 8B {e8["all"][k]:.3f}  4B {e4["all"][k]:.3f}')
gold = load_jsonl('data/processed/en/multifc_test.jsonl') + load_jsonl('data/copies/en/test.jsonl')
avt = load_jsonl('data/processed/en/averitec_test.jsonl')
for size, mf, av in (('8B', 'results/F4/preds_llm_judge.jsonl', 'results/F4/preds_avt_llm_judge.jsonl'),
                     ('4B', 'results/H7/qwen4b/preds_llm_judge4b.jsonl', 'results/H7/qwen4b/preds_avt_llm_judge4b.jsonl')):
    m = compute_all(join_gold(load_preds(mf), gold)[0]); a = compute_all(join_gold(load_preds(av), avt)[0])
    print(f"LLM judge {size}: MultiFC macro-F1 {m['macro_f1']:.3f} ESR {m['esr']:.3f} unrelated {m['unrelated_rate']:.3f} "
          f"paraphrase {m['paraphrase_consistency']:.3f} | AVeriTeC macro-F1 {a['macro_f1']:.3f}")

## 7. Push to `himasri/H7-qwen4b-swap` (fill in your GitHub email)

In [ ]:
GIT_NAME = 'Himasri'
GIT_EMAIL = 'YOUR_GITHUB_EMAIL'
token = userdata.get('GH_TOKEN')
!rm -f results/H7/qwen4b/*.partial
!git config user.name "{GIT_NAME}"
!git config user.email "{GIT_EMAIL}"
!git add results/H7/qwen4b
!git status --short | head
!git commit -q -m "[H7] Qwen3-4B 4-bit vs 8B: claim extraction, explanations + faithfulness, LLM judge"
!git push -q https://{token}@github.com/himaallu/Group-Project.git himasri/H7-qwen4b-swap && echo pushed

When it says `pushed`: Runtime → Disconnect and delete runtime.